In [1]:
import ee
ee.Initialize(project='mineral-prospectivity-zim')
print("Project's GEE initialized.")

ROI = ee.Geometry.Rectangle([25.24, -22.42, 33.07, -15.61])
print(f'Region of interest: {ROI.getInfo()}')

# Date ranges
START_DATE_DRY = '2019-05-01'
END_DATE_DRY   = '2024-10-31'
START_DATE     = '2019-01-01'
END_DATE       = '2024-12-31'

# Band mapping
BANDS = ['B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12']
BAND_NAMES = ['Blue', 'Green', 'Red', 'NIR', 'RE1', 'RE2', 'RE3', 'RE4', 'SWIR1', 'SWIR2']

print('Bands..... ')
print('***********************')
for i, band in enumerate(BAND_NAMES):
    print(f"{band} , {BANDS[i]}")
print(f'Dry season dates: {START_DATE_DRY} to {END_DATE_DRY}')
print(f'Full period dates: {START_DATE} to {END_DATE}')

Project's GEE initialized.
Region of interest: {'type': 'Polygon', 'coordinates': [[[25.24, -22.42], [33.07, -22.42], [33.07, -15.61], [25.24, -15.61], [25.24, -22.42]]]}
Bands..... 
***********************
Blue , B2
Green , B3
Red , B4
NIR , B8
RE1 , B5
RE2 , B6
RE3 , B7
RE4 , B8A
SWIR1 , B11
SWIR2 , B12
Dry season dates: 2019-05-01 to 2024-10-31
Full period dates: 2019-01-01 to 2024-12-31


In [2]:
# ---------------------------------------------------------------------
# 1. Join & Cloud Masking function (reused for both composites)
# ---------------------------------------------------------------------
join = ee.Join.saveFirst('cloud_prob')
condition = ee.Filter.equals(
    leftField='system:index',
    rightField='system:index'
)

def mask_clouds_s2(image):
    """Mask clouds using the S2 cloud probability (<20%) and scale to 0-1."""
    cloud_prob = ee.Image(image.get('cloud_prob'))
    cloud_mask = cloud_prob.select('probability').lt(20)
    sr_bands = image.select(BANDS).divide(10000)
    return (sr_bands.updateMask(cloud_mask)
                    .rename(BAND_NAMES)
                    .copyProperties(image, ['system:time_start']))

# ---------------------------------------------------------------------
# 2. Core function to build a median composite for any period
# ---------------------------------------------------------------------
def build_s2_composite(start_date, end_date, roi, month_range=None):
    """
    Builds a median Sentinel-2 composite.

    Args:
        start_date (str): Start date (YYYY-MM-DD).
        end_date (str): End date (YYYY-MM-DD).
        roi (ee.Geometry): Region of interest.
        month_range (tuple, optional): (start_month, end_month) e.g., (5, 10).
                                       Filters images to specific calendar months.
    Returns:
        ee.Image: Median composite clipped to ROI.
    """
    # Load surface reflectance and cloud probability collections
    s2_sr = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterBounds(roi)
             .filterDate(start_date, end_date)
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80)))

    s2_clouds = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
                 .filterBounds(roi)
                 .filterDate(start_date, end_date))

    # Join the two collections
    joined = ee.ImageCollection(join.apply(s2_sr, s2_clouds, condition))

    # Optionally filter by calendar month (e.g., May-Oct for dry season)
    if month_range:
        joined = joined.filter(ee.Filter.calendarRange(month_range[0], month_range[1], 'month'))

    # Apply cloud masking and compute median
    masked = joined.map(mask_clouds_s2)
    return masked.median().clip(roi)

In [3]:
# Build the dry-season composite (May–October)
composite_dry = build_s2_composite(
    start_date=START_DATE_DRY,
    end_date=END_DATE_DRY,
    roi=ROI,
    month_range=(5, 10)  # explicitly keep only May-Oct
)

print('Dry season bands:', composite_dry.bandNames().getInfo())

# Export task
task_dry = ee.batch.Export.image.toAsset(
    image=composite_dry,
    description='sentinel2_Zimbabwe_dry_season',
    assetId='projects/mineral-prospectivity-zim/assets/sentinel2_Zimbabwe_dry_season',
    region=ROI,
    scale=20,
    crs='EPSG:4326',
    maxPixels=1e13
)
task_dry.start()
print('✅ Dry season export submitted.')

Dry season bands: ['Blue', 'Green', 'Red', 'NIR', 'RE1', 'RE2', 'RE3', 'RE4', 'SWIR1', 'SWIR2']
✅ Dry season export submitted.


In [5]:
# Build the full-year composite (all months)
composite_full = build_s2_composite(
    start_date=START_DATE,
    end_date=END_DATE,
    roi=ROI,
    month_range=None  # no month filter → uses the entire date range
)

print('Full‑year bands:', composite_full.bandNames().getInfo())

# Export task
task_full = ee.batch.Export.image.toAsset(
    image=composite_full,
    description='sentinel2_Zimbabwe_median_2019_2024',
    assetId='projects/mineral-prospectivity-zim/assets/sentinel2_Zimbabwe_median_2019_2024',
    region=ROI,
    scale=20,
    crs='EPSG:4326',
    maxPixels=1e13
)
task_full.start()
print('✅ Full‑year export submitted.')

Full‑year bands: ['Blue', 'Green', 'Red', 'NIR', 'RE1', 'RE2', 'RE3', 'RE4', 'SWIR1', 'SWIR2']
✅ Full‑year export submitted.
